# Project - Generate Synthetic Data 

The Goal of this project is to generate a variety of Synthetic Data
1. Using Open Source models
2. Test a variety of models
3. Test a variety of prompts
4. Create a Gradio UI for this product
5. To be run in Google Collab as my system was running out of memory

In [ ]:
try:
    !pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
except Exception:
    print("Not in Colab. pip install failed")

In [ ]:
# Importing the necessary libraries

# import os
# import requests
from IPython.display import Markdown, display, update_display
# from openai import OpenAI
# from google.colab import drive

# from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
from huggingface_hub import login
import torch
import gradio as gr

In [ ]:
# Sign in to HuggingFace Hub

# hf_token = userdata.get('HF_TOKEN')
# login(hf_token, add_to_git_credential=True)

# from huggingface_hub import login
# login(huggingface_api_key)

# If running in Colab and using secrets:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token, add_to_git_credential=True)
except Exception:
    print("Not in Colab or token fetch failed. Please login manually if needed.")

MODEL_OPTIONS = [
    "meta-llama/Llama-3.2-3B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v0.4",
    # Add more models here as you wish
]

OpenAI API Key exists and begins sk-proj-
OpenRouter API Key exists and begins sk-or-v1
HuggingFace API Key exists and begins hf_XuzAY


In [ ]:
SYSTEM_PROMPT = """
    You are a helpful assistant that generates fake but realistic employee records for a tech startup.
"""

USER_PROMPT_TEMPLATE = (
    "Generate {n} fake but realistic employee records for a tech startup as csv with these columns:\n"
    "EmployeeID, Name, Age, Gender, Department, JobTitle, StartDate, Salary, Status, ManagerID, Email, Details, History.\n"
    "Make each row plausible. Do not include anything except the CSV data.\n"
)
# messages = [
#     {"role": "system", "content": system_prompt},
#     {"role": "user", "content": user_prompt}
# ]

# Caching models and tokenizers for performance
model_cache = {}
tokenizer_cache = {}



In [ ]:
def generate_employees(model_name, n_employees):
    # Compose the prompt
    user_prompt = USER_PROMPT_TEMPLATE.format(n=n_employees)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    # Load tokenizer and model, with quantization if supported
    if model_name not in tokenizer_cache:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer_cache[model_name] = tokenizer
    else:
        tokenizer = tokenizer_cache[model_name]

    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

    if model_name not in model_cache:
        # Load with quantization when possible
        try:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
                bnb_4bit_quant_type="nf4"
            )
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="auto",
                quantization_config=quant_config
            )
        except Exception as e:
            # Fallback to non-quantized if quantized load fails
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="auto"
            )
        model_cache[model_name] = model
    else:
        model = model_cache[model_name]

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=2000,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Try to extract only the CSV (remove the prompt, if echoed)
    # import re
    # match = re.search(r"(EmployeeID.*?[\r\n]+(?:.+[\r\n]+)+)", response)
    # csv_output = match.group(1).strip() if match else response.strip()

    # return csv_output

    # Trying to match the number of employees to what was requested
    import re
    match = re.search(r"(EmployeeID[^\n\r]*[\n\r].+)", response, re.DOTALL)
    csv_text = match.group(1).strip() if match else response.strip()

    lines = csv_text.strip().splitlines()
    if not lines or len(lines) < 2:
        return "No employee data generated. Try again."

    header = lines[0]
    data_rows = [l for l in lines[1:] if l.strip() and not l.lower().startswith("employeeid")]
    # Keep only first n_employees rows
    final_lines = [header] + data_rows[:n_employees]
    return "\n".join(final_lines)




In [ ]:
# Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# 🧑‍💻 LLM-powered Employee Data Generator")
    with gr.Row():
        model_dropdown = gr.Dropdown(label="LLM Model", choices=MODEL_OPTIONS, value=MODEL_OPTIONS[0])
        n_employees_slider = gr.Slider(label="Number of Employees", minimum=1, maximum=50, value=10, step=1)
    gen_button = gr.Button("Generate Data")
    output_box = gr.Textbox(label="CSV Output", show_copy_button=True, lines=15)
    gen_button.click(
        generate_employees,
        inputs=[model_dropdown, n_employees_slider],
        outputs=[output_box]
    )



In [ ]:
demo.launch(inbrowser=True)
